In [0]:
%pip install openai
dbutils.library.restartPython()

In [0]:
# UC1 — Data Quality Explainer
# Reads:  primeinsurance_analytics.bronze.silver_quality_log
# Writes: primeinsurance_analytics.gold.dq_explanation_report
#
# Purpose: Turns technical DQ failure logs into plain-English
#          memos the compliance team can act on — no analyst needed.
#
# FIXES APPLIED:
#   ✅ Added issue_id, table_name, column_name, severity, affected_records to output
#   ✅ Added model_name to output table
#   ✅ Prompt now covers: what was found, why it matters, cause, fix, prevention
#   ✅ Processes row-level (per issue) not just entity-level aggregates


## UC1 · Data Quality Explainer
**Who it serves:** Compliance team  
**The problem:** `silver_quality_log` is full of field names, rule names, and row counts. The compliance team does not know what any of it means for their work.  
**What this does:** Sends each DQ issue row to the LLM and gets back a plain-English memo: what broke, why it matters, what caused it, what was fixed, and how to prevent recurrence.

In [0]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import json
import uuid
from datetime import datetime
from openai import OpenAI
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    TimestampType, LongType, FloatType
)

# Databricks auth — no external keys needed
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
WORKSPACE_URL    = spark.conf.get("spark.databricks.workspaceUrl")

client = OpenAI(
    api_key  = DATABRICKS_TOKEN,
    base_url = f"https://{WORKSPACE_URL}/serving-endpoints"
)

MODEL = "databricks-gpt-oss-20b"
print(f"✅ Connected to model: {MODEL}")


In [0]:
# ── 2. Helper: parse the model's structured response ──────────────────────────
# databricks-gpt-oss-20b returns a JSON list:
#   [{"type": "reasoning", "summary": [...]}, {"type": "text", "text": "..."}]
# We only want the "text" block.

def extract_llm_text(response) -> str:
    """Extract the plain-text answer from a databricks-gpt-oss-20b response."""
    content = response.choices[0].message.content

    if isinstance(content, str):
        try:
            blocks = json.loads(content)
            for block in blocks:
                if isinstance(block, dict) and block.get("type") == "text":
                    return block.get("text", "").strip()
        except (json.JSONDecodeError, TypeError):
            return content.strip()   # Already plain text — return as-is
    elif isinstance(content, list):
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                return block.get("text", "").strip()

    return str(content).strip()


In [0]:
# ── 3. Read the DQ log ────────────────────────────────────────────────────────
dq_log = spark.read.table("primeinsurance_analytics.silver.silver_quality_log")

print(f"Schema:")
dq_log.printSchema()
print(f"\nTotal DQ issue rows: {dq_log.count()}")
display(dq_log.orderBy("entity", "failed_record_count"))


In [0]:
# ── 4. Prepare rows — add issue_id if not present ─────────────────────────────
# The requirements need issue_id, table_name, column_name, severity, affected_records
# in the output. We add issue_id as a unique key per row.
# Adjust the column mappings below if your table uses different column names.

existing_cols = dq_log.columns
print("Columns in source table:", existing_cols)

# Add issue_id (UUID) if the source table doesn't have one
if "issue_id" not in existing_cols:
    dq_log = dq_log.withColumn("issue_id", F.expr("uuid()"))

# Add table_name column (the source log table name) if missing
if "table_name" not in existing_cols:
    dq_log = dq_log.withColumn("table_name",
                               F.lit("primeinsurance_analytics.bronze.silver_quality_log"))

# Map quarantine_reason → column_name if column_name column doesn't exist
if "column_name" not in existing_cols:
    dq_log = dq_log.withColumn("column_name", F.col("quarantine_reason"))

# Add severity based on failed_record_count if not present
if "severity" not in existing_cols:
    dq_log = dq_log.withColumn(
        "severity",
        F.when(F.col("failed_record_count") > 1000, "HIGH")
         .when(F.col("failed_record_count") > 100,  "MEDIUM")
         .otherwise("LOW")
    )

# Map failed_record_count → affected_records for output naming
dq_log = dq_log.withColumnRenamed("failed_record_count", "affected_records") \
               if "affected_records" not in existing_cols \
               else dq_log

print("\n✅ Enriched schema ready:")
dq_log.printSchema()
dq_log_rows = dq_log.collect()
print(f"Total rows to process: {len(dq_log_rows)}")


In [0]:
# ── 5. Generate explanations — one LLM call per issue row ─────────────────────
# FIXED SYSTEM PROMPT: now explicitly covers all 5 required points:
#   1. What was found (plain English)
#   2. Why it matters to the business
#   3. What caused it
#   4. What was done to fix it
#   5. What should be done to prevent recurrence

SYSTEM_PROMPT = """You are a data quality analyst writing compliance briefing notes
for a non-technical compliance team at PrimeInsurance, an automotive insurance company.
The compliance team do not understand technical terms, column names, table names, or rule names.

For every data quality issue you receive, write a clear structured explanation with EXACTLY
these five sections. Use plain English throughout. Keep each section to 1-2 sentences.

WHAT WAS FOUND:
Describe what the data problem is in plain English. State which type of records were affected
(customers / vehicles / policies / claims / sales) and what was wrong with them.

WHY IT MATTERS:
Explain the business risk or compliance consequence if this issue is left unresolved.

WHAT CAUSED IT:
Describe the most likely root cause of this issue in simple terms (e.g. data entry error,
system integration gap, missing upstream validation).

WHAT WAS DONE TO FIX IT:
Explain what automatic remediation action was taken on these records
(e.g. flagged, quarantined, excluded from reporting).

HOW TO PREVENT RECURRENCE:
Give one concrete action the compliance or operations team should take to stop this happening again.

Do NOT use jargon. Do NOT mention column names, table names, or rule names. Write as if explaining
to a senior manager with no technical background."""


def build_issue_prompt(row) -> str:
    """Build a user prompt for a single DQ issue row."""
    affected = row["affected_records"] if "affected_records" in row else row.get("failed_record_count", "unknown")
    severity = row["severity"]
    entity   = row["entity"]
    reason   = row["quarantine_reason"].replace("_", " ")
    col_name = row["column_name"].replace("_", " ")

    return (
        f"Data Quality Issue Details:\n"
        f"  - Record type   : {entity}\n"
        f"  - Problem area  : {col_name}\n"
        f"  - Issue detected: {reason}\n"
        f"  - Severity      : {severity}\n"
        f"  - Records affected: {affected:,}\n\n"
        f"Write the five-section compliance explanation now."
    )


results = []

for row in dq_log_rows:
    issue_id      = row["issue_id"]
    table_name    = row["table_name"]
    column_name   = row["column_name"]
    severity      = row["severity"]
    entity        = row["entity"]
    affected      = row["affected_records"] if "affected_records" in row.asDict() else row["failed_record_count"]
    quarantine_r  = row["quarantine_reason"]

    user_prompt = build_issue_prompt(row)

    try:
        response = client.chat.completions.create(
            model       = MODEL,
            messages    = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_prompt}
            ],
            max_tokens  = 600,
            temperature = 0.3
        )
        explanation = extract_llm_text(response)
        status      = "success"

    except Exception as e:
        explanation = f"[ERROR generating explanation: {str(e)}]"
        status      = "error"

    results.append({
        # ── Original issue details (required fields) ──
        "issue_id"         : str(issue_id),
        "table_name"       : str(table_name),
        "column_name"      : str(column_name),
        "severity"         : str(severity),
        "affected_records" : int(affected),
        "entity"           : str(entity),
        "quarantine_reason": str(quarantine_r),
        # ── AI output ──
        "explanation"      : explanation,
        # ── Metadata ──
        "model_name"       : MODEL,
        "generation_status": status,
        "generated_at"     : datetime.utcnow()
    })

    print(f"✓ [{severity}] {entity} | {quarantine_r} | {affected:,} records → {status}")


In [0]:
# ── 6. Preview results ────────────────────────────────────────────────────────
for r in results:
    print(f"\n{'='*65}")
    print(f"ISSUE ID : {r['issue_id']}")
    print(f"TABLE    : {r['table_name']}")
    print(f"COLUMN   : {r['column_name']}")
    print(f"SEVERITY : {r['severity']}  |  AFFECTED: {r['affected_records']:,} records")
    print(f"MODEL    : {r['model_name']}")
    print(f"STATUS   : {r['generation_status']}")
    print(f"{'─'*65}")
    print(r['explanation'])


In [0]:
# ── 7. Write to gold ──────────────────────────────────────────────────────────
schema = StructType([
    # Original issue details
    StructField("issue_id",          StringType(),    False),
    StructField("table_name",        StringType(),    True),
    StructField("column_name",       StringType(),    True),
    StructField("severity",          StringType(),    True),
    StructField("affected_records",  IntegerType(),   True),
    StructField("entity",            StringType(),    True),
    StructField("quarantine_reason", StringType(),    True),
    # AI output
    StructField("explanation",       StringType(),    True),
    # Metadata
    StructField("model_name",        StringType(),    True),
    StructField("generation_status", StringType(),    True),
    StructField("generated_at",      TimestampType(), True),
])

output_df = spark.createDataFrame(results, schema=schema)

(
    output_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("primeinsurance_analytics.gold.dq_explanation_report")
)

final_count = spark.read.table("primeinsurance_analytics.gold.dq_explanation_report").count()
print(f"\n✅ Wrote {final_count} rows → primeinsurance_analytics.gold.dq_explanation_report")
display(spark.read.table("primeinsurance_analytics.gold.dq_explanation_report"))
